In [ ]:
import pandas as pd 
import numpy as np 
from importlib import reload
import datetime 
from PDU_Func import IO_Data, ActivitySummary,Authentification


In [ ]:
UserAuthDict = {
  "data": {
    "user_name": "Admin(TEST)",
    "user_id": "MIH",
    "user_email": "irsyadhbtlh96@gmail.com",
    "user_company_name": "PDU",
    "user_cid": "",
    "status": "Authorized"
  },
  "verification": "verified"
}
WellInfoDict = {
  "cid": 9,
  "wid": 146,
  "WellName": "SMP AAE-05",
  "RigName": "8    PRA#1\nName: rig_name, dtype: string",
  "ActiveDate": "Timestamp('2021-06-28 00:00:00')",
  "EndDate": "Timestamp('2021-08-31 00:00:00')"
}
UserDateRange={
  "StartDate": datetime.date(2021, 7, 31),
  "StartTime": datetime.time(0, 0),
  "EndDate": datetime.date(2021, 8, 1),
  "EndTime": datetime.time(23, 59)
}
print("Start Time")
print(str(UserDateRange['StartDate']) + " " + str(UserDateRange['StartTime']))
print(str(UserDateRange['EndDate']) + " " + str(UserDateRange['EndTime']))

## get Activity Log

In [ ]:

ActivityLog = IO_Data.DomeGetData(WellInfoDict, UserDateRange, table_type="ActivityLogTable")
ActivityLog

## get Realtime Sensor

In [ ]:
WellInfoDict = {
  "cid": 9, # KS ORKA
  "wid": 146, # SMP AAE-05
  "WellName": "SMP AAE-05",
  "RigName": "8    PRA#1\nName: rig_name, dtype: string",
  "ActiveDate": "Timestamp('2021-06-28 00:00:00')",
  "EndDate": "Timestamp('2021-08-31 00:00:00')"
}
DateTimeRange = {
  "StartDate": "2021-07-28",
  "StartTime": "00:00:00",
  "EndDate": "2021-07-29",
  "EndTime": "23:59:00"
}
reload(IO_Data)
Out = IO_Data.DomeGetRealtimeSensorData(WellInfoDict, DateTimeRange)
Out

In [ ]:
reload(ActivitySummary)
ActivitySummaryTable = ActivitySummary.ActivitySummaryTable(WellInfoDict, UserDateRange)
OutTest = ActivitySummaryTable.getActivitySummary(ActivityLog, Out)
OutTest = ActivitySummary.SubActivityMapping(OutTest)
OutTest

In [ ]:


def GenerateDuration_DF_v4(RealTime_DB, DrillActivityList='default'):
    if DrillActivityList=='default':
        DrillActivityList = ["DRILLING FORMATION", 'CIRCULATE HOLE CLEANING','CONNECTION','DRILL OUT CEMENT',]


    RealTime_DB['dt'] = pd.to_datetime(RealTime_DB['dt'])
    RealTime_DB["LABEL_All"] =RealTime_DB['SubActivity'].astype(str)+'--'+RealTime_DB['Activity'].astype(str)

    # TODO , a funtion that check the latest hole depth
    Hole_Depth = 0
    list_dict_out = []
    for k,DF_Temp in RealTime_DB.groupby((RealTime_DB['LABEL_All'].shift() != RealTime_DB['LABEL_All']).cumsum()):

        # if not(("Hole_Depth" in locals()) or ("Hole_Depth" in globals())):
        #     Hole_Depth = 0

        list_temp = []

        # date = DF_Temp.head(1)['date'].values[0]

        # Time_start = pd.to_datetime(date + " " + DF_Temp.head(1)['time'].values[0])
        # Time_end_temp = DF_Temp.tail(1)['time'].values[0]
        # Time_end = pd.to_datetime(DF_Temp.tail(1)['date'].values[0] + " " + str((datetime.strptime(Time_end_temp, "%H:%M:%S") + timedelta(seconds=5)).time()))

        StartDateTime = DF_Temp['dt'].iloc[0]
        EndDateTime = DF_Temp['DateTime'].iloc[-1]
        # Time_start = DF_Temp.head(1)['time'].values[0]
        # Time_end_temp = DF_Temp.tail(1)['time'].values[0]
        # Time_end = str((datetime.strptime(Time_end_temp, "%H:%M:%S") + timedelta(seconds=5)).time())

        LABEL_Activity = DF_Temp['Activity'].iloc[0]
        LABEL_SubActivity = DF_Temp['SubActivity'].iloc[0]
        section_data = DF_Temp['Section Size'].iloc[0]
        pic_data = DF_Temp['PIC'].iloc[0]
        InSlip_Treshold_data = DF_Temp['In-Slip Threshold'].iloc[0]
        remarks_data = DF_Temp['remarks'].iloc[0]

        # Duration(minutes)
        Duration = (StartDateTime - EndDateTime).total_seconds() / 60.0 


        Hole_Depth_Temp = DF_Temp['md'].max()
        if float(Hole_Depth_Temp)>= float(Hole_Depth):
            Hole_Depth = Hole_Depth_Temp
        
        Bit_Depth = DF_Temp['bitdepth'].mean()

        MeterageDrilling = np.nan

        RotateDrilling = np.nan
        SlideDrilling = np.nan
        ReamingTime = np.nan
        ConnectionTime = np.nan

        OnBottomHours = np.nan
        StandDuration = np.nan

        if LABEL_Activity in DrillActivityList:
            MeterageDrilling = round((DF_Temp['md'].max()-DF_Temp['md'].min()),2)

        if LABEL_SubActivity == "Rotary Drilling":
            RotateDrilling = Duration

        if LABEL_SubActivity == "Slide Drilling":
            SlideDrilling = Duration

        if LABEL_SubActivity == "Reaming":
            ReamingTime = Duration
        if LABEL_SubActivity == "Connection":
            ConnectionTime = Duration


        list_dict_out.append(
            {
            'date':date,
            'Time_start':Time_start,
            'Time_end':Time_end,
            'date_time':date_time,
            'Duration(minutes)':Duration,
            'Hole Depth(max)':Hole_Depth,
            'Bit Depth(mean)':Bit_Depth,
            "Meterage(m)(Drilling)": MeterageDrilling,
            'RotateDrilling':RotateDrilling,
            'Slide Drilling':SlideDrilling,
            'ReamingTime':ReamingTime,
            'ConnectionTime':ConnectionTime,
            'On Bottom Hours':OnBottomHours,
            'Stand Duration': StandDuration,
            'LABEL_SubActivity': LABEL_SubActivity,
            "LABEL_Activity": LABEL_Activity,
            "PIC":pic_data,
            "Remarks":remarks_data,
            "Section":section_data,

            }
        )
    Duration_DB = pd.DataFrame.from_dict(list_dict_out, orient='columns')
    Duration_DB = Duration_DB.astype(
        {
            'date':'datetime64',
            'Duration(minutes)':'float',
            'Hole Depth(max)':'float',
            'Bit Depth(mean)':'float',
            "Meterage(m)(Drilling)":'float',
            'RotateDrilling':'float',
            'Slide Drilling':'float',
            'ReamingTime':'float',
            'ConnectionTime':'float',
            'On Bottom Hours':'float',
            'Stand Duration':'float',
            "LABEL_SubActivity":'str', 
            "LABEL_Activity":'str',
            "PIC":'str',
            "Remarks":'str',
            "Section":'str',
        }
    )

    listSetDecimals = [
            'Duration(minutes)',
            'Hole Depth(max)',
            'Bit Depth(mean)',
            "Meterage(m)(Drilling)",
            'RotateDrilling',
            'Slide Drilling',
            'ReamingTime',
            'ConnectionTime',
            'On Bottom Hours',
            'Stand Duration',
    ]
    for ColumnName in listSetDecimals:
        # print(ColumnName)
        Duration_DB[ColumnName] = Duration_DB[ColumnName].round(decimals=2)

    # rounded_df = df.round(decimals=2)
    return Duration_DB

GenerateDuration_DF_v4(OutTest)

In [ ]:
# GenerateDuration
## Aggregate 1

# as it is
# datetime start 
# datetime end 
# date 
# well name
# activity
# sub-activity
# section size
# PIC
# in-slip threshold
# remarks

# 
# calculate on the fly
# Duration 
# Hole Depth 
# Bit Depth 
# Drilling Meterage
# Rotate Drilling Time 
# Slide Drilling Time
# Reaming Time
# Connection Time


# LabelStand
## Aggregate 2
# Connection Activity
# On Bottom Duration 
# Stand Duration 


In [ ]:
OutTest.columns

In [ ]:
test = pd.DataFrame(
    {
    "a":[1,2,np.nan,4],
    "b":[1,2,3,4],
    }
)
test.mean()

In [ ]:
7/4

## Realtime Sensor Download parallel

In [ ]:
from datetime import datetime
import requests
import json
import time
from Page import Welcome
from datetime import datetime
import pandas as pd
def DomeGetRealtimeSensorData(WellInfoDict, UserDateRange):
    # UserDateRange = {
    #         "StartDate": UserDateRange['StartDate'].strftime('%Y-%m-%d'),
    #         "StartTime": UserDateRange['StartTime'].strftime('%H:%M:%S'),
    #         "EndDate": UserDateRange['EndDate'].strftime('%Y-%m-%d'),
    #         "EndTime": UserDateRange['EndTime'].strftime('%H:%M:%S'),
    #         }

    # TODO: uncomment the code below if Realtime Data is ready
    Data_params ={
        "wid" : int(WellInfoDict['wid']),
        "start" : str(UserDateRange['StartDate']) + " " + str(UserDateRange['StartTime']),
        "end" : str(UserDateRange['EndDate']) + " " + str(UserDateRange['EndTime']),
    }
    

    column_list = ["dt", "date", "time", "bitdepth", "md", "blockpos", "rop", "hklda", "woba", "torqa", "rpm", "stppress", "mudflowin",]

    WellData = (
        (
            # requests.get("https://pdumitradome.id/dome_api/rtdc/rtdc/get_data", data=json.dumps(Data_params))
            requests.get("http://khansadev.xyz/dome_api/rtdc/get_data", data=json.dumps(Data_params))
        ).text
    )
    Realtime_DF = pd.json_normalize(json.loads(WellData), record_path='result')
    Realtime_DF['dt'] = Realtime_DF['dt'].astype('datetime64[ns]')
    start = datetime.strptime(Data_params['start'], '%Y-%m-%d %H:%M:%S')
    end = datetime.strptime(Data_params['end'], '%Y-%m-%d %H:%M:%S')
    mask = (Realtime_DF['dt'] > start) & (Realtime_DF['dt'] <= end)
    filtered_DF = Realtime_DF[mask]
    return filtered_DF[column_list]


UserDateRange = {
  "StartDate": "2023-04-28",
  "StartTime": "01:10:00",
  "EndDate": "2023-04-29",
  "EndTime": "23:10:00"
}
WellInfoDict = {
  "cid": 74,
  "wid": 243,
  "WellName": "IJN 2-1",
  "RigName": "2    EPI-9\nName: rig_name, dtype: string",
  "ActiveDate": "Timestamp('2023-04-05 00:00:00')",
  "EndDate": "Timestamp('2023-05-31 00:00:00')"
}

# DomeGetRealtimeSensorData(WellInfoDict, UserDateRange)

In [ ]:
def splitDateTime(UserDateRange, hours=0.5):
    startDateTimeStr = UserDateRange['StartDate'] + ' ' + UserDateRange['StartTime']
    endDateTimeStr = UserDateRange['EndDate'] + ' ' + UserDateRange['EndTime']


    startDateTime = pd.Timestamp(startDateTimeStr)
    endDateTime = pd.Timestamp(endDateTimeStr)
    deltaTime = pd.Timedelta(hours=hours)

    date_ranges = pd.date_range(start=startDateTime, end=endDateTime.floor("H"), freq=deltaTime)

    date_list = date_ranges.tolist() + [endDateTime]
    StartDateTimeList,EndDateTimeList = date_list[:-1],date_list[1:]
    return StartDateTimeList,EndDateTimeList
StartDateTimeList,EndDateTimeList = splitDateTime(UserDateRange, hours=0.5)
for i,j in zip(StartDateTimeList,EndDateTimeList):
    print(str(i) + " - " + str(j))
    # for i,j in zip(date_list[:-1],date_list[1:]):
    #     print(str(i) + " - " +str(j))

In [ ]:
import time
import numpy as np
from tqdm import tqdm
def DomeGetRealtimeSensorData_v2(WellInfoDict, UserDateRange, hours=0.5):
    # UserDateRange = {
    #         "StartDate": UserDateRange['StartDate'].strftime('%Y-%m-%d'),
    #         "StartTime": UserDateRange['StartTime'].strftime('%H:%M:%S'),
    #         "EndDate": UserDateRange['EndDate'].strftime('%Y-%m-%d'),
    #         "EndTime": UserDateRange['EndTime'].strftime('%H:%M:%S'),
    #         }
    StartDateTimeList,EndDateTimeList = splitDateTime(UserDateRange, hours=hours)
    # TODO: uncomment the code below if Realtime Data is ready
    column_list = ["dt", "date", "time", "bitdepth", "md", "blockpos", "rop", "hklda", "woba", "torqa", "rpm", "stppress", "mudflowin",]
    Realtime_List = []
    time_elapsed = []
    i = 0
    StartEndList=list(zip(StartDateTimeList,EndDateTimeList))

    for StartEnd in tqdm(((StartEndList))):
        StartDateTime,EndDateTime = StartEnd
        loopStartTime = time.time()
        Data_params ={
            "wid" : int(WellInfoDict['wid']),
            "start" : str(StartDateTime),
            "end" : str(EndDateTime),
        }
        # print(Data_params)

        WellData = json.loads((
            (
                # requests.get("https://pdumitradome.id/dome_api/rtdc/rtdc/get_data", data=json.dumps(Data_params))
                requests.get("http://khansadev.xyz/dome_api/rtdc/get_data", data=json.dumps(Data_params))
            ).text
        ))
        if WellData['status'] != 200:
            raise KeyError (WellData['message'])
        
        Realtime_DF_temp = pd.json_normalize((WellData), record_path='result')
        Realtime_DF_temp['dt'] = Realtime_DF_temp['dt'].astype('datetime64[ns]')
        Realtime_List.append(Realtime_DF_temp)
        loopEndTime = time.time()
        i=i+1
        time_elapsed.append(loopEndTime-loopStartTime)
        # print(f"Iteration {i}: {(loopEndTime-loopStartTime):.4f} seconds")
    Realtime_DF = pd.concat(Realtime_List, axis=0, ignore_index=True)

    start = str(UserDateRange['StartDate']) + " " + str(UserDateRange['StartTime'])
    end = str(UserDateRange['EndDate']) + " " + str(UserDateRange['EndTime'])
    mask = (Realtime_DF['dt'] > start) & (Realtime_DF['dt'] <= end)
    filtered_DF = Realtime_DF[mask]
    print( np.mean(time_elapsed))
    return filtered_DF[column_list]
starttime = time.time()
DomeGetRealtimeSensorData_v2(WellInfoDict, UserDateRange, hours=0.5)
print(time.time() - starttime)

In [ ]:
from multiprocessing.pool import ThreadPool
from functools import partial

def DomeGetRealtimeSensorDataSingleDate(WellInfoDict, StartDateTime, EndDateTime):
    # UserDateRange = {
    #         "StartDate": UserDateRange['StartDate'].strftime('%Y-%m-%d'),
    #         "StartTime": UserDateRange['StartTime'].strftime('%H:%M:%S'),
    #         "EndDate": UserDateRange['EndDate'].strftime('%Y-%m-%d'),
    #         "EndTime": UserDateRange['EndTime'].strftime('%H:%M:%S'),
    #         }

    # TODO: uncomment the code below if Realtime Data is ready
    column_list = ["dt", "date", "time", "bitdepth", "md", "blockpos", "rop", "hklda", "woba", "torqa", "rpm", "stppress", "mudflowin",]

    # for StartDateTime,EndDateTime in zip(StartDateTimeList,EndDateTimeList):
    Data_params ={
        "wid" : int(WellInfoDict['wid']),
        "start" : str(StartDateTime),
        "end" : str(EndDateTime),
    }

    WellData = json.loads((
        (
            # requests.get("https://pdumitradome.id/dome_api/rtdc/rtdc/get_data", data=json.dumps(Data_params))
            requests.get("http://khansadev.xyz/dome_api/rtdc/get_data", data=json.dumps(Data_params))
        ).text
    ))
    if WellData['status'] != 200:
        raise KeyError (WellData['message'])
    
    Realtime_DF = pd.json_normalize((WellData), record_path='result')
    Realtime_DF['dt'] = Realtime_DF['dt'].astype('datetime64[ns]')

        # print(f"Iteration {i}: {(loopEndTime-loopStartTime):.4f} seconds")


    # start = str(UserDateRange['StartDate']) + " " + str(UserDateRange['StartTime'])
    # end = str(UserDateRange['EndDate']) + " " + str(UserDateRange['EndTime'])
    # start = datetime.strptime(Data_params['start'], '%Y-%m-%d %H:%M:%S')
    # end = datetime.strptime(Data_params['end'], '%Y-%m-%d %H:%M:%S')
    # mask = (Realtime_DF['dt'] > start) & (Realtime_DF['dt'] <= end)
    # filtered_DF = Realtime_DF[mask]

    return Realtime_DF[column_list]

def DomeGetRealtimeSensorData_MultiCPU(WellInfoDict, UserDateRange,hours=0.5)

    StartDateTimeList,EndDateTimeList = splitDateTime(UserDateRange, hours=0.5)
    partial_get_data = partial(DomeGetRealtimeSensorDataSingleDate, WellInfoDict)

    with ThreadPool(processes=8) as pool:
        Realtime_List = pool.starmap(partial_get_data, zip(StartDateTimeList,EndDateTimeList))
    Realtime_DF = pd.concat(Realtime_List, axis=0, ignore_index=True)
    start = str(UserDateRange['StartDate']) + " " + str(UserDateRange['StartTime'])
    end = str(UserDateRange['EndDate']) + " " + str(UserDateRange['EndTime'])
    mask = (Realtime_DF['dt'] > start) & (Realtime_DF['dt'] <= end)
    filtered_DF = Realtime_DF[mask]
    return filtered_DF


In [ ]:
import multiprocessing
from multiprocessing.pool import ThreadPool

# Get the number of available CPU cores
num_cores = multiprocessing.cpu_count()
num_cores

In [ ]:
a= [1,2,3]
b=[4,3,2]
out = list(zip(a,b))
out

In [ ]:
StartEnd,EndStart = out[0]

In [ ]:
EndStart

## fill the gap activity summary

In [ ]:
import pandas as pd
import numpy as np

# create sample dataframe
df = pd.DataFrame({
    'startDateTime': ['2023-05-01 09:00:00', '2023-05-01 10:00:00', '2023-05-01 11:00:00', '2023-05-01 12:00:00'],
    'endDateTime': ['2023-05-01 09:10:00', '2023-05-01 10:50:00', '2023-05-01 11:30:00', '2023-05-01 13:00:00'],
    'Activity': ['Activity A', 'Activity B', 'Activity E', 'Activity C']
})

# sort dataframe by startDateTime column
df = df.sort_values('startDateTime')

# calculate time difference between rows
df['TimeDiff'] = (pd.to_datetime(df['startDateTime']) - pd.to_datetime(df['endDateTime'].shift())).dt.total_seconds()

# determine where there is a gap between activities
df['Gap'] = np.where((df['Activity'].shift() != 'Gap') & (df['TimeDiff'] > 0), 'Yes', 'No')
df

In [ ]:
end_gap = list(df.loc[df['Gap'] == 'Yes', 'startDateTime'])
start_gap = list(df.loc[(df['Gap'] == 'Yes').shift(), 'endDateTime'])


In [ ]:
import pandas as pd

# create sample dataframe
df = pd.DataFrame({
    'startDateTime': ['2023-05-01 09:00:00', '2023-05-01 10:00:00', '2023-05-01 11:00:00', '2023-05-01 12:00:00'],
    'endDateTime': ['2023-05-01 09:10:00', '2023-05-01 10:50:00', '2023-05-01 11:10:00', '2023-05-01 13:00:00'],
    'Activity': ['Activity A', 'Activity B', 'Activity E', 'Activity C']
})

# sort dataframe by startDateTime column
df = df.sort_values('startDateTime')

# calculate time difference between rows
df['TimeDiff'] = (pd.to_datetime(df['startDateTime']) - pd.to_datetime(df['endDateTime'].shift())).dt.total_seconds()

# create start and end gap lists
start_gap = []
end_gap = []
for index, row in df.iterrows():
    if row['TimeDiff'] > 0:
        start_gap.append(df.loc[index-1, 'endDateTime'])
        end_gap.append(row['startDateTime'])

# print start_gap and end_gap
print("start_gap:", start_gap)
print("end_gap:", end_gap)


In [ ]:
def getGapList(df, StartDateTime='startDateTime', EndDateTime='endDateTime'):
    df = df.sort_values(StartDateTime)

    # calculate time difference between rows
    df['TimeDiff'] = (pd.to_datetime(df[StartDateTime]) - pd.to_datetime(df[EndDateTime].shift())).dt.total_seconds()

    # create start and end gap lists
    start_gap = []
    end_gap = []
    for index, row in df.iterrows():
        if row['TimeDiff'] > 0:
            start_gap.append(df.loc[index-1, EndDateTime])
            end_gap.append(row[StartDateTime])
    return start_gap, end_gap
    # # print start_gap and end_gap
    # print("start_gap:", start_gap)
    # print("end_gap:", end_gap)
getGapList(df)